<a href="https://colab.research.google.com/github/aligreo/TriEncoder-Unet-Project/blob/main/mslesseg_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MSLesSeg Data Reading Plan for TriEncoder U-Net

## Required Data Format
* Each sample must be a dictionary with keys: **t1**, **t2**, **flair**, **mask_label**.
* Keep modalities separate; do not concatenate them into **image**.

## Dataset Layout
* Train: **MSLesSeg Dataset/train/P*/T*/**
* Test: **MSLesSeg Dataset/test/P*/**
* Each valid case must contain files ending with **_T1.nii.gz**, **_T2.nii.gz**, **_FLAIR.nii.gz**, **_MASK.nii.gz**.

## Previous Mistakes
* The old notebook matched labels using broad terms including **"T3"**, which confused timepoint folders with masks.
* Because of that, 9 training cases selected a T1 image as the label, such as **P1/T3/P1_T3_T1.nii.gz**.
* The old collector returned 51 training cases, but the correct timepoint-aware count is 87 training cases.
* The test split was okay because it is flat: each test patient folder directly contains **T1**, **T2**, **FLAIR**, and **MASK**.
* The old transform concatenated modalities into one **image** tensor, but TriEncoder U-Net needs separate inputs: **t1**, **t2**, **flair**, plus **mask_label**.
* A previous attempted notebook edit failed while the **F:** drive was full, leaving **mslesseg_preprocessing.ipynb** truncated to 0 bytes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!uv pip install SimpleITK monai nibabel

In [ ]:
import os
import random
import warnings
import shutil
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    NormalizeIntensityd, ConcatItemsd, DeleteItemsd, EnsureTyped,
    CropForegroundd, Lambdad, Resized
)
from monai.data import PersistentDataset, DataLoader, pad_list_data_collate
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
mslesseg_train_data = "/content/drive/MyDrive/mslesseg folder/MSLesSeg Dataset/train"
mslesseg_test_data = "/content/drive/MyDrive/mslesseg folder/MSLesSeg Dataset/test"
CACHE_DIR = "/content/cache_mslesseg"

In [ ]:
def binarize_label(x):
    return (x > 0.5).astype(np.float32)

def find_files_in_dir(path):
    """Finds T1, T2, FLAIR, and MASK files in a directory using exact suffix matching."""
    if not os.path.exists(path): return None
    files = os.listdir(path)
    item = {}
    
    mapping = {
        "t1": "_T1.nii.gz",
        "t2": "_T2.nii.gz",
        "flair": "_FLAIR.nii.gz",
        "mask_label": "_MASK.nii.gz"
    }
    
    for key, suffix in mapping.items():
        found = [f for f in files if f.upper().endswith(suffix.upper())]
        if found:
            best_file = sorted(found, key=len)[0]
            item[key] = os.path.join(path, best_file)
        else:
            return None
    return item

def collect_dataset(root_path, is_train=True):
    """Collects MSLesSeg data, handling nested timepoint directories for training."""
    data_list = []
    if not os.path.exists(root_path): return []

    for subject in sorted(os.listdir(root_path)):
        subj_path = os.path.join(root_path, subject)
        if not os.path.isdir(subj_path): continue
        
        if is_train:
            # Train: MSLesSeg Dataset/train/P*/T*/
            timepoints = [d for d in os.listdir(subj_path) if os.path.isdir(os.path.join(subj_path, d))]
            for tp in sorted(timepoints):
                tp_path = os.path.join(subj_path, tp)
                item = find_files_in_dir(tp_path)
                if item:
                    item["subject"] = f"{subject}_{tp}"
                    data_list.append(item)
        else:
            # Test: MSLesSeg Dataset/test/P*/
            item = find_files_in_dir(subj_path)
            if item:
                item["subject"] = subject
                data_list.append(item)

    return data_list

def create_transforms():
    keys = ["flair", "t1", "t2", "mask_label"]
    return Compose([
        LoadImaged(keys=keys),
        EnsureChannelFirstd(keys=keys),
        Orientationd(keys=keys, axcodes="RAS"),
        Spacingd(keys=keys, pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "bilinear", "bilinear", "nearest")),
        CropForegroundd(keys=keys, source_key="flair"),
        Resized(keys=keys, spatial_size=(96, 96, 96), mode=("trilinear", "trilinear", "trilinear", "nearest")),
        Lambdad(keys="mask_label", func=binarize_label),
        NormalizeIntensityd(keys=["flair", "t1", "t2"], nonzero=True, channel_wise=True),
        EnsureTyped(keys=keys),
    ])

def get_loaders(train_files, test_files, cache_dir):
    random.seed(42)
    train_ds = PersistentDataset(data=train_files, transform=create_transforms(), cache_dir=cache_dir)
    test_ds = PersistentDataset(data=test_files, transform=create_transforms(), cache_dir=cache_dir)

    return (
        DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=pad_list_data_collate),
        DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=pad_list_data_collate)
    )

In [ ]:
train_files = collect_dataset(mslesseg_train_data, is_train=True)
test_files = collect_dataset(mslesseg_test_data, is_train=False)

print(f"Total valid training cases (timepoints): {len(train_files)}")
print(f"Total valid testing cases: {len(test_files)}")

if train_files and test_files:
    train_loader, test_loader = get_loaders(train_files, test_files, CACHE_DIR)
    print("Data loaders successfully initialized.")
else:
    print("Error: Missing valid training or testing cases.")

In [ ]:
def visualize_sample(sample_dict, slice_idx=None):
    # Load label to find the best slice
    label_img = nib.load(sample_dict['mask_label']).get_fdata()
    
    if slice_idx is None:
        # Find slice with maximum lesion area
        slice_sums = label_img.sum(axis=(0, 1))
        if slice_sums.max() > 0:
            current_slice = int(np.argmax(slice_sums))
        else:
            current_slice = label_img.shape[2] // 2
    else:
        current_slice = slice_idx
        
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    modalities = ["flair", "t1", "t2", "mask_label"]
    
    flair_slice = None
    label_slice = label_img[:, :, current_slice]

    for i, mod in enumerate(modalities):
        img = nib.load(sample_dict[mod]).get_fdata()
        slice_data = img[:, :, current_slice]
        
        if mod == "flair":
            flair_slice = slice_data
            
        cmap = "hot" if mod == "mask_label" else "gray"
        axes[i].imshow(slice_data, cmap=cmap)
        axes[i].set_title(f"{sample_dict.get('subject', 'case')} | {mod.upper()} | Slice {current_slice}")
        axes[i].axis("off")

    # 5th Plot: Overlay
    axes[4].imshow(flair_slice, cmap="gray")
    masked_label = np.ma.masked_where(label_slice == 0, label_slice)
    axes[4].imshow(masked_label, cmap="autumn", alpha=0.7, vmin=0, vmax=1)
    axes[4].set_title(f"{sample_dict.get('subject', 'case')} | Overlay | Slice {current_slice}")
    axes[4].axis("off")

    plt.tight_layout()
    plt.show()

if "train_files" in globals() and train_files:
    visualize_sample(train_files[0])